# pod-clone 환경을 Colab에 맞추기 — uv + Python 3.11.9

Colab 런타임 커널은 Python 3.12지만, **uv로 standalone Python 3.11.9 venv**를 만들어
이미지(pod-clone)와 **Python 패치 버전까지 정확히** 일치시킵니다.

> ⚠️ 이 3.11.9 환경은 **venv/서브프로세스**로 실행됩니다(노트북 셀의 커널 자체는 3.12).
> 실제 코드는 `/content/py311/bin/python your_script.py` 또는 `uv run`으로 돌리세요.
> GPU가 필요하면 **Runtime → Change runtime type → T4** 로 바꾼 뒤 실행하세요
> (NVIDIA 드라이버는 시스템 레벨이라 venv에서도 GPU가 잡힙니다).

## 1. uv 설치 + Python 3.11.9 venv 생성

In [ ]:
!pip install -q uv
!uv venv --python 3.11.9 /content/py311
!/content/py311/bin/python --version

## 2. requirements 파일 작성 (이미지에서 추출한 131개 패키지)

In [ ]:
%%writefile requirements-image.txt
accelerate==1.2.0
aiohappyeyeballs==2.6.2
aiohttp==3.14.1
aiosignal==1.4.0
annotated-doc==0.0.4
annotated-types==0.7.0
anthropic==0.84.0
anyio==4.14.0
attrs==26.1.0
backoff==2.2.1
bcrypt==5.0.0
build==1.5.0
certifi==2026.6.17
cffi==2.0.0
charset-normalizer==3.4.7
chromadb==1.3.6
click==8.4.1
cryptography==49.0.0
datasets==3.2.0
dill==0.3.8
distro==1.9.0
docstring_parser==0.18.0
durationpy==0.10
filelock==3.29.0
filetype==1.2.0
flatbuffers==25.12.19
frozenlist==1.8.0
fsspec==2024.9.0
google-auth==2.55.0
google-genai==1.75.0
googleapis-common-protos==1.75.0
grpcio==1.81.1
h11==0.16.0
httpcore==1.0.9
httptools==0.8.0
httpx==0.28.1
huggingface-hub==0.26.5
idna==3.18
importlib_resources==7.1.0
Jinja2==3.1.6
jiter==0.15.0
jsonpatch==1.33
jsonpointer==3.1.1
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
kubernetes==36.0.2
langchain==1.2.10
langchain-anthropic==1.3.4
langchain-chroma==1.0.0
langchain-core==1.2.17
langchain-google-genai==4.2.1
langchain-openai==1.1.10
langgraph==1.0.10
langgraph-checkpoint==4.0.1
langgraph-prebuilt==1.0.8
langgraph-sdk==0.3.9
langsmith==0.7.13
markdown-it-py==4.2.0
MarkupSafe==3.0.3
mdurl==0.1.2
mmh3==5.2.1
mpmath==1.3.0
multidict==6.7.1
multiprocess==0.70.16
networkx==3.6.1
numpy==1.26.4
oauthlib==3.3.1
onnxruntime==1.27.0
openai==2.26.0
opentelemetry-api==1.42.1
opentelemetry-exporter-otlp-proto-common==1.42.1
opentelemetry-exporter-otlp-proto-grpc==1.42.1
opentelemetry-proto==1.42.1
opentelemetry-sdk==1.42.1
opentelemetry-semantic-conventions==0.63b1
orjson==3.11.9
ormsgpack==1.12.2
overrides==7.7.0
packaging==26.2
pandas==3.0.3
pillow==12.2.0
posthog==5.4.0
propcache==0.5.2
protobuf==6.33.6
psutil==7.2.2
pyarrow==24.0.0
pyasn1==0.6.3
pyasn1_modules==0.4.2
pybase64==1.4.3
pycparser==3.0
pydantic==2.12.5
pydantic_core==2.41.5
Pygments==2.20.0
PyPika==0.51.1
pyproject_hooks==1.2.0
python-dateutil==2.9.0.post0
python-dotenv==1.2.2
PyYAML==6.0.3
referencing==0.37.0
regex==2026.5.9
requests==2.34.2
requests-oauthlib==2.0.0
requests-toolbelt==1.0.0
rich==15.0.0
rpds-py==2026.5.1
safetensors==0.8.0
shellingham==1.5.4
six==1.17.0
sniffio==1.3.1
sympy==1.14.0
tenacity==9.1.4
tiktoken==0.13.0
tokenizers==0.21.1
torch==2.4.0
torchaudio==2.4.0
torchvision==0.19.0
tqdm==4.68.3
transformers==4.49.0
typer==0.26.7
typing-inspection==0.4.2
typing_extensions==4.15.0
urllib3==2.7.0
uuid_utils==0.16.2
uvicorn==0.49.0
uvloop==0.22.1
watchfiles==1.2.0
websocket-client==1.9.0
websockets==16.0
xxhash==3.7.0
yarl==1.24.2
zstandard==0.25.0

## 3. 3.11.9 venv에 설치 (uv pip)

uv라서 빠릅니다. **torch는 Pod과 동일하게 cu124 빌드로 맞춥니다**(NVIDIA T4 기준 — CUDA 12.4 / cuDNN 9.1).
CPU 런타임에서만 쓸 거면 cu124 줄은 생략해도 됩니다(불필요한 ~2GB).

In [ ]:
!uv pip install --python /content/py311/bin/python -r requirements-image.txt
# Pod과 동일한 torch 2.4.0+cu124 로 덮어쓰기 (NVIDIA GPU/T4 기준). CPU 전용이면 이 줄 생략 가능.
!uv pip install --python /content/py311/bin/python \
    torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 \
    --index-url https://download.pytorch.org/whl/cu124
print('설치 완료 (Python 3.11.9 venv · torch 2.4.0+cu124)')

## 4. 검증 — 3.11.9 venv에서 버전 일치 + import + GPU

venv 인터프리터로 직접 실행하므로 ABI 충돌 없이 깨끗하게 확인됩니다.

In [ ]:
%%writefile verify_env.py
import sys, importlib.metadata as md
# 기준: pod-clone 이미지. torch는 +cu124 로컬 라벨이 붙으므로 base 버전(+ 앞)으로 비교한다.
EXPECTED = {
    'torch': '2.4.0', 'torchvision': '0.19.0', 'torchaudio': '2.4.0',
    'transformers': '4.49.0', 'tokenizers': '0.21.1', 'huggingface-hub': '0.26.5',
    'datasets': '3.2.0', 'accelerate': '1.2.0', 'numpy': '1.26.4', 'pydantic': '2.12.5',
    'langchain': '1.2.10', 'langchain-core': '1.2.17', 'langchain-openai': '1.1.10',
    'langchain-anthropic': '1.3.4', 'langgraph': '1.0.10', 'langgraph-checkpoint': '4.0.1',
    'langsmith': '0.7.13', 'openai': '2.26.0', 'anthropic': '0.84.0', 'chromadb': '1.3.6',
}
print('python', sys.version.split()[0], '(목표 이미지: 3.11.9)\n')
ok = bad = 0
for pkg, want in EXPECTED.items():
    try:
        got = md.version(pkg)
    except md.PackageNotFoundError:
        got = None
    base = got.split('+')[0] if got else got
    match = base == want
    ok += match
    bad += not match
    print(f"{'✅' if match else '❌'} {pkg:22s} 기대 {want:12s} 실제 {got}")
print(f'\n일치 {ok} / 불일치 {bad}')
import torch, transformers, langchain, langgraph
print('torch', torch.__version__, '| built cuda', torch.version.cuda,
      '| cuda?', torch.cuda.is_available(),
      '| device:', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
print('OK: 핵심 라이브러리 import 성공 (Python', sys.version.split()[0], ')')

In [ ]:
!/content/py311/bin/python verify_env.py